# **Feature Store & Batch Inference: AI-Powered Apple Leaf Specialist**

* **Name**: Aktham Almomani
* **Group**: 7

## **Introduction**

This notebook builds a small, end-to-end feature store workflow for the apple leaf project. It creates a feature group for image metadata, ingests a demo table, validates online and offline stores, runs Athena queries, and demonstrates batch inference with a PyTorch model using JSONL input.

Here're the main steps in this notebooks:

* Stand up a Feature Group for image-level features.
* Ingest and validate records in online and offline stores.
* Query features via Athena for analysis.
* Run a SageMaker Batch Transform job with an existing model artifact.



## **Setup and Parameters**

First, let's setup using SageMaker SDK to obtain session, role, region, and default S3 bucket. Bucket paths are used for the offline store and Athena outputs

In [ ]:
import boto3, sagemaker, pandas as pd, time, uuid
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum

sess   = sagemaker.Session()
role   = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()

fg_name = "apple_image_meta_fg"
offline_s3_uri = f"s3://{bucket}/feature-store/{fg_name}/offline"

# here let's define the schema:
feature_definitions = [
    FeatureDefinition(feature_name="image_id",        feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="s3_uri",          feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="split",           feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="label",           feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="brightness_bin",  feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="avg_r",           feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="avg_g",           feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="avg_b",           feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="event_time",      feature_type=FeatureTypeEnum.STRING),
]



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## **Define Feature Group**

Here let's create `apple_image_meta_fg` with schema fields such as `image_id`, `s3_uri`, `split`, `label`, `brightness_bin`, `avg_r`, `avg_g`, `avg_b`, and `event_time`. Configure `OnlineStore` enabled and `OfflineStore` to S3. Handle SDK version differences for `create()` signatures.

* **brightness_bin:** A categorical bucket that encodes overall image brightness (luminance).

* **avg_r**: Mean intensity of the `red` channel across all pixels in the image.

* **avg_g**: Mean intensity of the `green` channel across all pixels.

* **avg_b**: Mean intensity of the `blue` channel across all pixels.



In [ ]:
fg = FeatureGroup(name=fg_name, sagemaker_session=sess)

try:
    fg.delete()
except Exception:
    pass

try:
    fg.feature_definitions = feature_definitions
except Exception:
    pass

try:
      fg.create(
        feature_definitions,
        record_identifier_name="image_id",
        event_time_feature_name="event_time",
        role_arn=role,
        online_store_config={"EnableOnlineStore": True},
        offline_store_config={"S3StorageConfig": {"S3Uri": offline_s3_uri}},
    )
except TypeError:
    fg.create(
        s3_uri=offline_s3_uri,
        record_identifier_name="image_id",
        event_time_feature_name="event_time",
        role_arn=role,
        enable_online_store=True,
    )

print("Feature group created:", fg.describe()["FeatureGroupName"])

Feature group created: apple_image_meta_fg


## **Ingest Demo Records**

Here let's build a tiny DataFrame with two sample images and matching columns, then call `fg.ingest(..., wait=True)` to load records.

In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc).isoformat()
demo = [
    {
        "image_id": str(uuid.uuid4()),
        "s3_uri": f"s3://{bucket}/apple/sample/healthy1.jpg",
        "split": "test",
        "label": "healthy",
        "brightness_bin": "bright",
        "avg_r": 0.52,
        "avg_g": 0.61,
        "avg_b": 0.49,
        "event_time": now,
    },
    {
        "image_id": str(uuid.uuid4()),
        "s3_uri": f"s3://{bucket}/apple/sample/rust1.jpg",
        "split": "test",
        "label": "rust",
        "brightness_bin": "dim",
        "avg_r": 0.42,
        "avg_g": 0.50,
        "avg_b": 0.55,
        "event_time": now,
    },
]
df = pd.DataFrame(demo)
cols = ["image_id","s3_uri","split","label","brightness_bin","avg_r","avg_g","avg_b","event_time"]
df = df[cols]
df




,image_id,s3_uri,split,label,brightness_bin,avg_r,avg_g,avg_b,event_time
0,91c1fb8d-b544-43a8-abfa-63247d39ca22,s3://sagemaker-us-east-1-533266958221/apple/sa...,test,healthy,bright,0.52,0.61,0.49,2025-10-18T04:38:48.672352+00:00
1,1a37c753-affc-46ff-b902-3fc20832620d,s3://sagemaker-us-east-1-533266958221/apple/sa...,test,rust,dim,0.42,0.50,0.55,2025-10-18T04:38:48.672352+00:00


## **Feature Group Status Check**

here let's poll `fg.describe()["FeatureGroupStatus"]` to ensure the group is Created before downstream operations.

In [ ]:
def wait_fg_active(fg, poll=5):
    while True:
        status = fg.describe()["FeatureGroupStatus"]
        print("FeatureGroupStatus:", status)
        if status in ("Created","CreateFailed"):
            break
        time.sleep(poll)

wait_fg_active(fg)



FeatureGroupStatus: Created


## **Query Offline Store with Athena**

Use Feature Store query helpers to create and run SQL against the Glue Data Catalog database and table backing the offline store. Fetch results as a DataFrame for quick validation and simple aggregations.

In [ ]:
# Here's let's query the offline store via Athena:
from sagemaker.feature_store.feature_store import FeatureStore

fs = FeatureStore(sagemaker_session=sess)
query = fs.create_query_with_feature_group(fg_name, table_name=None)
sql = f"SELECT image_id, label, brightness_bin, avg_r, avg_g, avg_b FROM {query.table_name} LIMIT 10"
athena = query.run(sql_query=sql, output_location=f"s3://{bucket}/athena-results/")
athena.as_dataframe().head()

In [ ]:
ingest_res = fg.ingest(data_frame=df, max_workers=4, wait=True)
print("Ingest complete.")

Ingest complete.


## **Online Store Read**

Verify near-real-time availability with `sagemaker-featurestore-runtime` `get_record` for a sample `image_id`, confirming keys and values returned from the online store.

In [ ]:
fs_rt = boto3.client("sagemaker-featurestore-runtime")

one_id = df.iloc[0]["image_id"]
resp = fs_rt.get_record(
    FeatureGroupName=fg.name,
    RecordIdentifierValueAsString=str(one_id)
)
print("Online GetRecord keys:", [f["FeatureName"] for f in resp.get("Record", [])])


Online GetRecord keys: ['image_id', 's3_uri', 'split', 'label', 'brightness_bin', 'avg_r', 'avg_g', 'avg_b', 'event_time']


In [ ]:
desc = fg.describe()
print("OfflineStoreConfig:", desc.get("OfflineStoreConfig"))
print("OfflineStoreStatus:", desc.get("OfflineStoreStatus"))
print("DataCatalogConfig:", desc.get("OfflineStoreConfig",{}).get("DataCatalogConfig"))
print("S3 uri:", desc.get("OfflineStoreConfig",{}).get("S3StorageConfig"))


OfflineStoreConfig: {'S3StorageConfig': {'S3Uri': 's3://sagemaker-us-east-1-533266958221/feature-store/apple_image_meta_fg/offline', 'ResolvedOutputS3Uri': 's3://sagemaker-us-east-1-533266958221/feature-store/apple_image_meta_fg/offline/533266958221/sagemaker/us-east-1/offline-store/apple_image_meta_fg-1760761969/data'}, 'DisableGlueTableCreation': False, 'DataCatalogConfig': {'TableName': 'apple_image_meta_fg_1760761969', 'Catalog': 'AwsDataCatalog', 'Database': 'sagemaker_featurestore'}}
OfflineStoreStatus: {'Status': 'Active'}
DataCatalogConfig: {'TableName': 'apple_image_meta_fg_1760761969', 'Catalog': 'AwsDataCatalog', 'Database': 'sagemaker_featurestore'}
S3 uri: {'S3Uri': 's3://sagemaker-us-east-1-533266958221/feature-store/apple_image_meta_fg/offline', 'ResolvedOutputS3Uri': 's3://sagemaker-us-east-1-533266958221/feature-store/apple_image_meta_fg/offline/533266958221/sagemaker/us-east-1/offline-store/apple_image_meta_fg-1760761969/data'}


In [ ]:
from sagemaker.feature_store.feature_group import AthenaQuery
import json

# Pull the real DB/table from describe()
desc = fg.describe()
db  = desc["OfflineStoreConfig"]["DataCatalogConfig"]["Database"]
tbl = desc["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]
print("Using:", db, ".", tbl)

# Create the helper and point it explicitly at the DB & table
query = fg.athena_query()
query.database   = db
query.table_name = tbl

# Pick an S3 location for Athena results:
output_loc = f"s3://{bucket}/athena/{tbl}/"

# Run a simple aggregation
sql = f'SELECT label, COUNT(*) AS n FROM "{db}"."{tbl}" GROUP BY label'
query.run(query_string=sql, output_location=output_loc)
query.wait()

# Fetch results as a dataframe
athena_df = query.as_dataframe()
athena_df



Using: sagemaker_featurestore . apple_image_meta_fg_1760761969


,label,n
0,rust,1
1,healthy,1


## **Batch Inference (Transform Job)**

Now let's prepare a JSONL file of base64-encoded images from `val.zip`, upload to S3, and launch a `PyTorchModel.transformer` job that uses your packaged `model.tar.gz` and `inference.py`. Wait for completion and capture S3 output location.

In [ ]:
# Batch Transform using the trained estimator:

import base64, os
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
from sagemaker.pytorch import PyTorchModel

sess   = sagemaker.Session()
role   = sagemaker.get_execution_role()
bucket = sess.default_bucket()
s3     = boto3.client("s3")

# let's start by getting val.zip:
zip_key = "apple/val.zip"
local_zip = "/tmp/val.zip"
s3.download_file(bucket, zip_key, local_zip)


In [ ]:
# Extract a handful of images:
extract_dir = Path("/tmp/val_images")
extract_dir.mkdir(exist_ok=True, parents=True)
with ZipFile(local_zip) as zf:
    zf.extractall(extract_dir)

# Collect some image files:
img_paths = []
for ext in (".jpg", ".jpeg", ".png"):
    img_paths += list(extract_dir.rglob(f"*{ext}"))
img_paths = img_paths[:20]

assert img_paths, "No images found after unzipping! Check the archive structure."

In [ ]:
# Build JSONL:
local_jsonl = Path("/tmp/batch_input.jsonl")
with local_jsonl.open("w") as f:
    for p in img_paths:
        with open(p, "rb") as im:
            b64 = base64.b64encode(im.read()).decode("utf-8")
        f.write(json.dumps({"b64": b64}) + "\n")

In [ ]:
# Upload JSONL to S3:
s3_in = sess.upload_data(str(local_jsonl), bucket=bucket, key_prefix="batch/input")
print("Uploaded to:", s3_in)

Uploaded to: s3://sagemaker-us-east-1-533266958221/batch/input/batch_input.jsonl


In [ ]:
# Finally let's run Batch Transform using your PyTorch inference code:

# Our trained model.tar.gz:
model_artifact = "s3://sagemaker-us-east-1-533266958221/apple/outputs/pytorch-training-2025-10-16-04-42-30-265/output/model.tar.gz"

pyt_model = PyTorchModel(
    model_data=model_artifact,
    role=role,
    framework_version="2.3",
    py_version="py311",
    entry_point="inference.py",
    source_dir="src",
    sagemaker_session=sess,
)

run_id = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
s3_out = f"s3://{bucket}/batch/output/{run_id}/"

transformer = pyt_model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=s3_out,
    strategy="SingleRecord",
)

transformer.transform(
    data=s3_in,
    content_type="application/json",
    split_type="Line",
)
transformer.wait()

print("Batch input :", s3_in)
print("Batch output:", s3_out)

/tmp/ipykernel_3627/440194388.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-533266958221/apple/outputs/pytorch-training-2025-10-16-04-42-30-265/output/model.tar.gz), script artifact (src), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-533266958221/pytorch-inference-2025-10-18-06-06-31-168/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-inference-2025-10-18-06-06-35-301
INFO:sagemaker:Creating transform job with name: pytorch-inference-2025-10-18-06-06-36-031


...............................['torchserve', '--start', '--model-store', '/.sagemaker/ts/models', '--ts-config', '/etc/sagemaker-ts.properties', '--log-config', '/opt/conda/lib/python3.11/site-packages/sagemaker_pytorch_serving_container/etc/log4j2.xml', '--models', 'model=/opt/ml/model']
2025-10-18T06:11:50,317 [WARN ] main org.pytorch.serve.util.ConfigManager - Your torchserve instance can access any URL to load models. When deploying to production, make sure to limit the set of allowed_urls in config.properties
2025-10-18T06:11:50,321 [INFO ] main org.pytorch.serve.servingsdk.impl.PluginsManager - Initializing plugins manager...
2025-10-18T06:11:50,402 [INFO ] main org.pytorch.serve.metrics.configuration.MetricConfiguration - Successfully loaded metrics configuration from /opt/conda/lib/python3.11/site-packages/ts/configs/metrics.yaml
2025-10-18T06:11:50,499 [INFO ] main org.pytorch.serve.ModelServer - 
Torchserve version: 0.11.0
TS Home: /opt/conda/lib/python3.11/site-packages
Cur